Introduction to GenAi



In [1]:
!pip install groq --quiet
import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
print("Libraries reply!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.8 MB/s eta 0:00:00
Libraries reply!


In [2]:
from groq import Groq
API_KEY="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"
client=Groq(api_key=API_KEY)
MODEL = "llama-3.1-8b-instant"
print(f"Groq client configured with model:{MODEL}")
print("Make sure API_KEY is replaced with your actual key!")

Groq client configured with model:llama-3.1-8b-instant
Make sure API_KEY is replaced with your actual key!


In [3]:
def ask_llm(
    user_message,
    system_message="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500
):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )

    return response.choices[0].message.content


# Test 1
test_response = ask_llm(
    "What is ELT in data engineering? Answer in exactly 2 sentences."
)

# Test 2
test_response2 = ask_llm(
    "Is GenAI and Data Engineering a good career in 2026?"
)

print("=== Response 1 ===")
print(test_response)

print("\n=== Response 2 ===")
print(test_response2)


=== Response 1 ===
ELT stands for Extract, Load, Transform, which is a data processing methodology used in data engineering and data warehousing. Unlike ETL (Extract, Transform, Load), which performs transformations before loading data into a target system, ELT executes transformations after loading the data, allowing for more flexible and efficient data processing.

=== Response 2 ===
As of 2026, GenAI (General Artificial Intelligence) and Data Engineering are indeed rapidly growing fields with high demand in the industry. Here's a brief overview of the career prospects in these fields:

**GenAI:**

1. **Growing demand**: GenAI is transforming various industries, including healthcare, finance, and education. As the technology advances, the demand for professionals who can develop and implement AI solutions will increase.
2. **High-paying jobs**: GenAI engineers, researchers, and developers are among the highest-paid professionals in the tech industry, with salaries ranging from $150,0

In [4]:
response_elt = ask_llm(
    "In 3 bullet points,explain how the Medallion Architecture"
    "(Bronze,Silver,Gold_layers) realtes to ELT pipeline.",
    system_message="You are a senior data engineering Instructor."
                   "Be concise and practical."
)
print('Medallion + ELT connection:')
print(response_elt)
print()
print('--- Token explanation ---')
print('Each word is roughly 1-2 tokens.')
print('The model above used approximately',len(response_elt.split())*1.3,'tokens.')
print('Llma-3.1-8b context window:8192 tokens(-6000 words per conversation)')

Medallion + ELT connection:
Here are 3 bullet points explaining the relationship between the Medallion Architecture and ELT pipeline:

• **Bronze Layer (Raw Data Store)**: The Bronze layer in the Medallion Architecture acts as the landing zone for raw data from the source systems. In an ELT (Extract, Load, Transform) pipeline, the Extract phase extracts data from the source systems, which is then loaded into the Bronze layer as raw, unprocessed data.

• **Silver Layer (Staging Area)**: The Silver layer is the staging area where data is transformed into a standardized format, making it easier to load into the Gold layer. In an ELT pipeline, the Transform phase transforms the raw data from the Bronze layer into a more usable format, which is then loaded into the Silver layer.

• **Gold Layer (Data Warehouse)**: The Gold layer in the Medallion Architecture is the data warehouse that stores processed and aggregated data for analytics and reporting. In an ELT pipeline, the Transform phase c

Prompt Engineering Experiments

In [5]:
# Zero-shot Prompting
zero_shot_response = ask_llm(
    "Extract the city name from this address: "
    "456 Brigade Road, Bangalore 560025, Karnataka, India"
)

print("Zero-shot Result:")
print(zero_shot_response)
print()


# Ambiguous Prompting
ambiguous_response = ask_llm(
    "Clean this data: ramesh kumar.4500.mumbai")

print("Ambiguous Zero-shot Result:")
print(ambiguous_response)
print()
print("Problem output format is unpredictable and not machine-parseable!")

Zero-shot Result:
The city name is Bangalore.

Ambiguous Zero-shot Result:
Here's the cleaned data:

- First name: Ramesh
- Last name: Kumar
- Value: 4500
- Location: Mumbai

The data is cleaned by separating the different pieces of information and identifying them as first name, last name, value, and location.

Problem output format is unpredictable and not machine-parseable!


In [6]:
few_shot_prompt="""
Convert employee text to JSON.Here are examples:
Input: RAMESH KUMAR,45000,mumbai
output:{"name":"Ramesh","salary":45000,"city":"Mumbai"}
Input:priya nair, 52000,Delhi
output:{"name":"Priya Nair","salary":52000,"city":"Delhi"}
Now convert this:
Input:ANANYA DAS, 30000,kolkata
Output:"""

few_shot_response=ask_llm(
    few_shot_prompt, temperature=0.0)
print('Few-Shot Result:')
print(few_shot_response)
print()
try:
  parsed = json.loads(few_shot_response.strip())
  print('Successfully parsed as JSON!')
  print(f'Name:{parsed["name"]},salary:{parsed['salary']},city:{parsed['city']}')
except json.JSONDecodeError:
        print('Parsing failed - model added extra text')
        print('Solution: add explicit instructions in the prompt')

Few-Shot Result:
To convert the employee text to JSON, we can use the following Python code:

```python
import json

def convert_to_json(employee_text):
    # Split the input string into individual values
    values = employee_text.split(',')

    # Create a dictionary with the given keys
    employee = {
        "name": values[0].strip().title(),
        "salary": int(values[1].strip()),
        "city": values[2].strip().title()
    }

    # Convert the dictionary to JSON
    json_output = json.dumps(employee, indent=4)

    return json_output

# Test the function
employee_text = "ANANYA DAS, 30000, kolkata"
print(convert_to_json(employee_text))
```

When you run this code, it will output:

```json
{
    "name": "Ananya Das",
    "salary": 30000,
    "city": "Kolkata"
}
```

This code works by splitting the input string into individual values using the comma as a delimiter. It then creates a dictionary with the given keys and assigns the corresponding values. Finally, it converts the 

In [7]:
same_question = (
    "Review this Python code and identify any issues:\n"
    "df['revenue'] = df['qty'] * df['price']\n"
    "result = df.groupby('dept').sum()"
)

# Without Role Prompting
generic_response = ask_llm(
    same_question,
    temperature=0.2
)

print("Without Role Prompting:")
print(generic_response[:300], "...")
print()

# With Role Prompting
role_response = ask_llm(
    same_question,
    system_message=(
        "You are a senior data engineer with 10 years of experience. "
        "Review code critically for production readiness, "
        "data type issues, performance concerns, and potential failures at scale."
    ),
    temperature=0.2
)

print("With Role Prompting (Senior Data Engineer):")
print(role_response[:400], "...")
print()

print("Notice: Role prompting produces more technical and actionable feedback.")

Without Role Prompting:
The provided Python code appears to be a part of a pandas DataFrame operation. However, there are a few potential issues that could be improved:

1. **Missing Import Statement**: The code assumes that pandas is imported, but it's not explicitly shown. Make sure to add `import pandas as pd` at the be ...

With Role Prompting (Senior Data Engineer):
**Code Review**

The provided Python code appears to be a basic data manipulation and aggregation operation using the pandas library. However, there are several potential issues that need to be addressed for production readiness:

```python
# Import necessary libraries
import pandas as pd

# Assuming df is a pandas DataFrame

# Potential issue 1: Division by zero
# If 'price' or 'qty' contains zer ...

Notice: Role prompting produces more technical and actionable feedback.


In [8]:
prompt ="Give me one creative name for a data analytics startup"
print('=== Temperature Experiment ===')
for temp in [0.0,0.5,1.0]:
  response = ask_llm(prompt, temperature=temp)
  print(f'Temperature={temp}:{response.strip()}')
  time.sleep(1)
print()
print('Observation:')
print('  temperature=0.0 -> same or very similar answer every run(deterministic)')
print('  temperature=0.5 -> some variation')
print('  temperature=1.0 more creative/varied, sometimes surprising')
print()
print('Rule for data engineering tasks: use temperature =0.0 or 0.1')
print('You need CONSISTENT, PARSEABLE ouput - not creative variation')


=== Temperature Experiment ===
Temperature=0.0:Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys a sense of innovation and forward-thinking, which is perfect for a data analytics startup.
Temperature=0.5:Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" is a combination of the words "next" and "axis," suggesting a connection to the next level of data analysis and a central axis of insights. This name implies that the startup is a hub for data-driven decision-making and innovative insights.
Temperature=1.0:Here's a creative name for a data analytics startup:

**"Nexa Insights"**

- "Nexa" is derived from the word "nexus," meaning a connection or relationship between things. This name suggests the connections and insights that your data analytics startup can provide to its clients.
- "Insights"

In [9]:
import json

invoice_text = (
    "Invoice #2024-001 from TECHWORLD SOLUTIONS "
    "dated 15th January 2024, Amount: Rs. 45000 for Laptop"
)

# Weak Prompt
weak_response = ask_llm(
    f"Clean this invoice data: {invoice_text}",
    temperature=0.3
)

print("WEAK PROMPT OUTPUT:")
print(weak_response)
print()

try:
    json.loads(weak_response)
    print("PARSEABLE: Yes")
except json.JSONDecodeError:
    print("PARSEABLE: No - Cannot load into DataFrame")

print("\n" + "=" * 50 + "\n")

# Strong Prompt
strong_system = """
You are a data extraction specialist for an accounting pipeline.

Extract invoice data and return ONLY a valid JSON object.
Do NOT include any explanation, preamble, or markdown formatting.
Return ONLY the JSON, nothing else.

JSON schema:
{
  "invoice_id": "",
  "vendor_name": "",
  "amount": 0,
  "currency": "INR",
  "invoice_date": "",
  "category": ""
}
"""

strong_response = ask_llm(
    f"Clean this invoice data: {invoice_text}",
    system_message=strong_system,
    temperature=0
)

print("STRONG PROMPT OUTPUT:")
print(strong_response)
print()

try:
    parsed = json.loads(strong_response.strip())

    print("PARSEABLE: Yes")
    print(f"Invoice ID : {parsed['invoice_id']}")
    print(f"Vendor     : {parsed['vendor_name']}")
    print(f"Amount     : {parsed['amount']}")
    print(f"Currency   : {parsed['currency']}")
    print(f"Date       : {parsed['invoice_date']}")
    print(f"Category   : {parsed['category']}")

except json.JSONDecodeError:
    print("PARSEABLE: No")

WEAK PROMPT OUTPUT:
**Cleaned Invoice Data:**

- **Invoice Number:** 2024-001
- **Date:** 15th January 2024
- **Invoice Issuer:** TECHWORLD SOLUTIONS
- **Description:** Laptop
- **Amount:** Rs. 45,000

**Note:** I have reformatted the date to a more standard format (DD/MM/YYYY) and added a description to the cleaned data. If you need any further assistance, please let me know.

PARSEABLE: No - Cannot load into DataFrame


STRONG PROMPT OUTPUT:
{
  "invoice_id": "2024-001",
  "vendor_name": "TECHWORLD SOLUTIONS",
  "amount": 45000,
  "currency": "INR",
  "invoice_date": "2024-01-15",
  "category": "Laptop"
}

PARSEABLE: Yes
Invoice ID : 2024-001
Vendor     : TECHWORLD SOLUTIONS
Amount     : 45000
Currency   : INR
Date       : 2024-01-15
Category   : Laptop
